# Notebook 5 - Analisis del Backtest Momentum

Analisis de resultados, robustez y sesgos de la estrategia ejecutada en Notebook 4.

## Imports y configuracion

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as fs

sns.set(style="whitegrid", context="notebook")

BENCH = "SPY"
FEE_RATE_BASE = 0.0023
MIN_FEE_BASE = 23.0

## Funciones auxiliares

In [ ]:
def compute_metrics(equity, benchmark=None, freq=252):
    eq = pd.Series(equity).dropna().sort_index()
    if len(eq) < 2:
        return {}
    r = eq.pct_change().dropna()
    yrs = len(r) / float(freq)
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1 / yrs) - 1 if yrs > 0 else np.nan
    vol = r.std(ddof=0) * np.sqrt(freq)
    sharpe = (r.mean() / r.std(ddof=0) * np.sqrt(freq)) if r.std(ddof=0) > 0 else np.nan
    dd = eq / eq.cummax() - 1
    out = {
        "total_return": eq.iloc[-1] / eq.iloc[0] - 1,
        "cagr": cagr,
        "vol": vol,
        "sharpe": sharpe,
        "max_drawdown": dd.min(),
    }
    if benchmark is not None:
        b = pd.Series(benchmark).dropna().sort_index()
        a = pd.concat([eq, b], axis=1, join="inner").dropna()
        if len(a) > 2:
            rs = a.iloc[:, 0].pct_change().dropna()
            rb = a.iloc[:, 1].pct_change().dropna()
            x = pd.concat([rs, rb], axis=1).dropna()
            if len(x) > 2:
                out["alpha_daily_mean"] = (x.iloc[:, 0] - x.iloc[:, 1]).mean()
                vb = np.var(x.iloc[:, 1])
                out["beta"] = np.cov(x.iloc[:, 0], x.iloc[:, 1])[0, 1] / vb if vb > 0 else np.nan
    return out


def monte_carlo_monkeys(monthly_returns, n_monos=3000, chunk_size=6, fee_rate=0.0023):
    r = pd.Series(monthly_returns).dropna().astype(float)
    if len(r) < max(24, chunk_size * 2):
        raise ValueError("No hay suficientes retornos mensuales para Monte Carlo por bloques.")

    v = r.values
    n = len(v)
    starts = np.arange(n - chunk_size + 1)
    drag = fee_rate / 12.0
    rows = []

    for _ in range(int(n_monos)):
        path = []
        while len(path) < n:
            st = np.random.choice(starts)
            path.extend(v[st:st + chunk_size].tolist())
        path = np.array(path[:n]) - drag
        wealth = np.cumprod(1 + path)
        yrs = n / 12.0
        cagr = wealth[-1] ** (1 / yrs) - 1 if yrs > 0 and wealth[-1] > 0 else np.nan
        vol = np.std(path, ddof=0) * np.sqrt(12)
        sharpe = (np.mean(path) / np.std(path, ddof=0) * np.sqrt(12)) if np.std(path, ddof=0) > 0 else np.nan
        rows.append((wealth[-1] - 1, cagr, vol, sharpe))

    return pd.DataFrame(rows, columns=["total_return", "cagr", "vol", "sharpe"])


def _load_csv_fallback(candidates):
    for p in candidates:
        try:
            df = pd.read_csv(p)
            if len(df) > 0:
                return df, p
        except Exception:
            continue
    raise FileNotFoundError(f"No se pudo cargar ninguno de estos archivos: {candidates}")

## Carga de outputs de Notebook 4

In [ ]:
equity_candidates = [
    "outputs/backtest_equity_curve.csv",
    "notebooks/outputs/backtest_equity_curve.csv",
    "backtest_equity_curve.csv",
]
trades_candidates = [
    "outputs/trades.csv",
    "notebooks/outputs/trades.csv",
    "trades.csv",
]
weights_candidates = [
    "outputs/weights_by_rebalance.csv",
    "notebooks/outputs/weights_by_rebalance.csv",
    "weights_by_rebalance.csv",
]


equity_df, equity_src = _load_csv_fallback(equity_candidates)
trades_df, trades_src = _load_csv_fallback(trades_candidates)
weights_df, weights_src = _load_csv_fallback(weights_candidates)

equity_df["date"] = pd.to_datetime(equity_df["date"], errors="coerce")
equity_df = equity_df.dropna(subset=["date"]).set_index("date").sort_index()

if "date" in trades_df.columns:
    trades_df["date"] = pd.to_datetime(trades_df["date"], errors="coerce")
if "rebalance_date" in weights_df.columns:
    weights_df["rebalance_date"] = pd.to_datetime(weights_df["rebalance_date"], errors="coerce")

print("equity source:", equity_src)
print("trades source:", trades_src)
print("weights source:", weights_src)

print("
Equity rows:", len(equity_df), "| Trades:", len(trades_df), "| Weights:", len(weights_df))
display(equity_df.head())
display(trades_df.head())

## Benchmark SPY y metricas principales

In [ ]:
s = equity_df.index.min().strftime("%Y-%m-%d")
e = (equity_df.index.max() + pd.Timedelta(days=3)).strftime("%Y-%m-%d")
spy = yf.download(BENCH, start=s, end=e, progress=False, auto_adjust=False)

if isinstance(spy.columns, pd.MultiIndex):
    if ("Close", BENCH) in spy.columns:
        bench_close = spy[("Close", BENCH)]
    elif ("Adj Close", BENCH) in spy.columns:
        bench_close = spy[("Adj Close", BENCH)]
    else:
        bench_close = spy.xs("Close", axis=1, level=0).iloc[:, 0]
else:
    bench_close = spy["Close"] if "Close" in spy.columns else spy["Adj Close"]

bench_close.index = pd.to_datetime(bench_close.index)
bench_close = bench_close.reindex(equity_df.index).ffill()

metrics_strat = compute_metrics(equity_df["equity"], bench_close)

bench_eq = bench_close / bench_close.dropna().iloc[0] * equity_df["equity"].iloc[0]
metrics_bench = compute_metrics(bench_eq)

print("Metricas estrategia:")
for k, v in metrics_strat.items():
    print(k, v)

print("
Metricas benchmark SPY (normalizado):")
for k, v in metrics_bench.items():
    print(k, v)

## Curvas, drawdown y retornos mensuales

In [ ]:
strat_eq = equity_df["equity"].copy()
bench_eq = bench_close / bench_close.dropna().iloc[0] * strat_eq.iloc[0]

plt.figure(figsize=(12, 5))
plt.plot(strat_eq.index, strat_eq.values, label="Momentum Strategy")
plt.plot(bench_eq.index, bench_eq.values, label="SPY")
plt.title("Equity Curve vs SPY")
plt.legend()
plt.tight_layout()
plt.show()

strat_dd = strat_eq / strat_eq.cummax() - 1
bench_dd = bench_eq / bench_eq.cummax() - 1

plt.figure(figsize=(12, 4))
plt.plot(strat_dd.index, strat_dd.values, label="Strategy DD")
plt.plot(bench_dd.index, bench_dd.values, label="SPY DD")
plt.title("Drawdown comparado")
plt.legend()
plt.tight_layout()
plt.show()

strat_m = strat_eq.resample("M").last().pct_change().dropna()
bench_m = bench_eq.resample("M").last().pct_change().dropna()

plt.figure(figsize=(10, 4))
sns.histplot(strat_m, bins=40, color="teal", alpha=0.7, label="Strategy", kde=True)
sns.histplot(bench_m, bins=40, color="orange", alpha=0.5, label="SPY", kde=True)
plt.legend()
plt.title("Distribucion de retornos mensuales")
plt.tight_layout()
plt.show()

## Sensibilidad a costes (robustez)

In [ ]:
if len(trades_df) == 0:
    print("No hay trades para sensibilidad de costes.")
    sensitivity_df = pd.DataFrame()
else:
    trades = trades_df.copy()
    trades["date"] = pd.to_datetime(trades["date"], errors="coerce")
    trades = trades.dropna(subset=["date", "notional", "fee"])    

    base_fee_by_day = trades.groupby("date")["fee"].sum().reindex(equity_df.index, fill_value=0.0)

    scenarios = [
        (0.0010, 10.0),
        (0.0023, 23.0),
        (0.0040, 23.0),
        (0.0023, 50.0),
    ]

    rows = []
    curves = {}

    for fr, mf in scenarios:
        alt_fee = trades["notional"].abs().apply(lambda n: max(fr * n, mf))
        alt_fee_by_day = alt_fee.groupby(trades["date"]).sum().reindex(equity_df.index, fill_value=0.0)

        fee_delta_cum = (alt_fee_by_day - base_fee_by_day).cumsum()
        eq_alt = equity_df["equity"] - fee_delta_cum
        eq_alt = eq_alt.clip(lower=1.0)

        m = compute_metrics(eq_alt, bench_close)
        rows.append({
            "fee_rate": fr,
            "min_fee": mf,
            "equity_final": eq_alt.iloc[-1],
            "cagr": m.get("cagr", np.nan),
            "sharpe": m.get("sharpe", np.nan),
            "max_drawdown": m.get("max_drawdown", np.nan),
        })

        curves[f"rate={fr},min={mf}"] = eq_alt

    sensitivity_df = pd.DataFrame(rows).sort_values(["fee_rate", "min_fee"])
    display(sensitivity_df)

    plt.figure(figsize=(12, 5))
    plt.plot(equity_df.index, equity_df["equity"], label="Base")
    for k, v in curves.items():
        plt.plot(v.index, v.values, label=k, alpha=0.8)
    plt.title("Sensibilidad de equity a costes")
    plt.legend()
    plt.tight_layout()
    plt.show()

## "Gran mentira" adaptada: monos Monte Carlo

In [ ]:
strat_m = equity_df["equity"].resample("M").last().pct_change().dropna()
bench_m = bench_close.resample("M").last().pct_change().dropna()

# Alineamos longitud para comparacion justa
n = min(len(strat_m), len(bench_m))
strat_m = strat_m.tail(n)
bench_m = bench_m.tail(n)

mc = monte_carlo_monkeys(bench_m, n_monos=3000, chunk_size=6, fee_rate=FEE_RATE_BASE)
strat_cagr = compute_metrics((1 + strat_m).cumprod())['cagr']

pct = (mc["cagr"] < strat_cagr).mean() * 100
print("CAGR estrategia:", strat_cagr)
print("Percentil vs monos (sobre CAGR):", round(pct, 2), "%")

display(mc.describe())

plt.figure(figsize=(10, 4))
sns.histplot(mc["cagr"].dropna(), bins=50, color="steelblue", kde=True)
plt.axvline(strat_cagr, color="red", linestyle="--", label="CAGR estrategia")
plt.title("Distribucion CAGR monos vs estrategia")
plt.legend()
plt.tight_layout()
plt.show()

## Seccion critica (obligatoria)

- **Impacto comisi?n m?nima 23$**: castiga especialmente ?rdenes peque?as y puede reducir fuerte el alpha neto.
- **Sesgo de supervivencia**: si el universo se construy? con activos supervivientes, el backtest puede estar inflado.
- **No look-ahead**: las se?ales vienen de NB3 (t-1 y anteriores) y la ejecuci?n en NB4 usa precios de D o anteriores.
- **Riesgo de overfitting**: los par?metros (6M/12M/top20/costes) pueden sobreajustar; por eso se reportan sensibilidad y Monte Carlo.
- **Rebalanceo irrealista residual**: no se incluye slippage real ni impacto de mercado, solo comisi?n expl?cita.
- **Comisiones pagadas**: ver `trades.csv` (suma de columna `fee`).